In [ ]:
from itertools import product

import matplotlib.pyplot as plt
import pandas as pd

from openplaces.api import get_admin1, get_admin2, get_admin_ids, read_entities
from openplaces.io import read_parquet
from openplaces.path import cache_path
from openplaces.recipe import get_recipe_by_id

DATASETS = {
    'parcels': 'US-NC_parcel-nconemap-2025',
    'nsi': 'US_building-usace-2022',
    'fema': 'US_building-fema-2023',
    'microsoft': 'US_building-microsoft-v2',
}

In [ ]:
def get_stats(gdf):
    gdf_stats = (gdf.notnull() & gdf.ne('')).sum().rename('n_values').to_frame()
    gdf_stats['frac_values'] = gdf_stats['n_values'] / len(gdf)
    gdf_stats.index.name = 'variable'
    gdf_stats = gdf_stats.join(gdf.describe(percentiles=[0.5]).T.drop(columns='count'))
    return gdf_stats


def get_value_counts(gdf, nmax=100):

    columns_to_count = []
    for column in gdf.columns:
        if (
            gdf[column].dtype not in ['object', 'category']
            or '_id_' in column
            or column.endswith('id')
            or '_date_' in column
            or column.endswith('_date')
            or column.endswith('city')
            or column.endswith('postal_code')
            or column.endswith('source')
        ):
            continue

        mask = gdf[column].notnull() & gdf[column].ne('')
        if mask.any():
            mean_duplicates = gdf[mask][column].duplicated(keep=False).mean()

            if mean_duplicates > 0.9:
                columns_to_count += [column]

    if not columns_to_count:
        return None

    value_counts_list = []
    for column in columns_to_count:
        mask = gdf[column].notnull() & gdf[column].ne('')
        value_counts = gdf[mask][column].value_counts().head(nmax).to_frame()
        value_counts.index.name = 'value'
        value_counts['variable'] = column
        value_counts_list += [value_counts]
    return pd.concat(value_counts_list).reset_index().set_index(['variable', 'value'])

In [ ]:
admin1 = get_admin1(
    'US-NC', geom=True, recipe=get_recipe_by_id('US_admin-nhgis-2020_admin1')
)

In [ ]:
admin2 = get_admin2(
    'US-NC', geom=True, recipe=get_recipe_by_id('US_admin-nhgis-2020_admin2')
)
admin_ids = list(admin2.index)

# Compute stats

In [ ]:
stats_lists = {k: [] for k in DATASETS.keys()}
value_counts_lists = {k: [] for k in DATASETS.keys()}
for admin_id, (dataset_key, dataset_recipe) in product(admin_ids, DATASETS.items()):
    dataset = read_entities(admin_id, dataset_recipe)

    dataset_stats = get_stats(dataset)
    dataset_stats.insert(0, 'admin_id', admin_id)
    stats_lists[dataset_key] += [dataset_stats]

    dataset_value_counts = get_value_counts(dataset)
    if dataset_value_counts is not None:
        dataset_value_counts.insert(0, 'admin_id', admin_id)
        value_counts_lists[dataset_key] += [dataset_value_counts]

stats = {
    dataset_key: pd.concat(stats_lists[dataset_key])
    .reset_index()
    .set_index(['variable', 'admin_id'])
    for dataset_key in DATASETS.keys()
}

value_counts = {
    dataset_key: pd.concat(value_counts_lists[dataset_key])
    .reset_index()
    .set_index(['variable', 'admin_id', 'value'])
    for dataset_key in DATASETS.keys()
    if len(value_counts_lists[dataset_key])
}

# Map variable presence

## By variable

In [ ]:
for dataset_key in DATASETS.keys():
    data = (
        stats[dataset_key]
        .groupby('variable')['frac_values']
        .mean()
        .mul(100)
        .loc[stats[dataset_key].index.get_level_values(0).unique()][::-1]
    )

    fig, ax = plt.subplots(figsize=(3, len(data) * 0.2))
    data.plot(kind='barh', ax=ax)
    ax.set_xlim(0, 100)
    ax.grid(axis='x', color='black', linewidth=0.2)
    ax.set_xlabel('average % non-null values (county)')
    ax.set_ylabel(None)
    ax.set_title(dataset_key)
    plt.plot()

## By county

In [ ]:
BINS = [0, 0.01, 0.03, 0.1, 0.2, 0.5, 0.8, 0.9, 0.97, 0.99, 1]

for dataset_key in DATASETS.keys():
    print(dataset_key)

    for variable in stats[dataset_key].index.get_level_values(0).unique():

        data = 1 - stats[dataset_key].loc[variable]['frac_values']
        if data.eq(0).all():
            continue

        fig, ax = plt.subplots(figsize=(9, 3))
        admin2.join(data).plot(
            'frac_values',
            ax=ax,
            scheme='user_defined',
            classification_kwds={'bins': BINS[1:], 'lowest': BINS[0]},
            legend=True,
            legend_kwds={
                'loc': 'center left',
                'bbox_to_anchor': (1, 0.5),
                'title': '% empty',
                'labels': [
                    f'{int(BINS[i]*100)} - {int(BINS[i+1]*100)}'
                    for i in range(len(BINS) - 1)
                ],
            },
            cmap='RdYlBu_r',
        )
        ax.set_title(f'{dataset_key}: {variable}')
        ax.axis('off')
        plt.show()

# Value counts

## Most frequent categories

In [ ]:
import matplotlib.patches as mpatches

for dataset_key in DATASETS.keys():
    most_frequent_categories = (
        value_counts[dataset_key]
        .query('count > 0')
        .sort_values('count', ascending=False)
        .reset_index()
        .drop_duplicates(['variable', 'admin_id'])
        .set_index(['variable', 'admin_id'])
    )
    variables = most_frequent_categories.index.get_level_values('variable').unique()

    for variable in variables:
        most_frequent_by_county = most_frequent_categories.loc[variable]['value']
        if not len(most_frequent_by_county):
            continue

        fig, ax = plt.subplots(figsize=(9, 3))

        admin2.join(most_frequent_by_county).plot(
            'value',
            ax=ax,
            cmap='tab20',
            legend=True,
            legend_kwds={'loc': 'upper center', 'bbox_to_anchor': (0.5, 0), 'ncols': 4},
        )
        admin1.boundary.plot(ax=ax, color='black', linewidth=0.1)

        ax.set_title(f'{dataset_key}: most frequent `{variable}`')
        ax.axis('off')
        plt.show()